# LeetCode 2682 — Find the Losers of the Circular Game bằng Python

Link đề: https://leetcode.com/problems/find-the-losers-of-the-circular-game/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Simulation

Skeleton LeetCode:

```python
class Solution:
    def circularGameLosers(self, n: int, k: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` người bạn ngồi thành vòng tròn, chuyền bóng theo luật "bước nhảy tăng dần" `k, 2k, 3k, ...` cho đến khi **có người nhận bóng lần thứ hai**; người **chưa từng** nhận bóng là "kẻ thua".
2. Biết **mô phỏng (simulation)**: làm đúng theo luật đề, từng bước một, và dừng đúng lúc.
3. Dùng phép **chia lấy dư `%`** để "đi vòng tròn" (khi vượt quá `n` thì quay lại đầu).
4. Dùng mảng đánh dấu `visited` (list `bool`) và `set` để ghi nhớ "ai đã nhận bóng".
5. Viết 4 cách: mô phỏng với `list` (tìm tuyến tính), mô phỏng với mảng đánh dấu, mô phỏng với `set` + hiệu hai tập hợp, và công thức số tam giác `k·t(t+1)/2 mod n`.
6. Tự kiểm tra bằng test ngẫu nhiên có seed, đối chiếu các cách với nhau và với tính tay.
7. Tránh các lỗi quen thuộc: lệch chỉ số 0/1, quên tăng bước nhảy, dừng sai thời điểm, quên sắp xếp kết quả.

## 2. Đề bài & ví dụ

Có `n` người bạn ngồi quanh một vòng tròn, đánh số từ `1` đến `n` **theo chiều kim đồng hồ**. Cụ thể, với `1 <= i < n`, bạn thứ `i` ngồi cạnh bạn thứ `i + 1`, và bạn thứ `n` ngồi cạnh bạn thứ `1`.

Luật chơi:

1. Bạn thứ `1` nhận bóng đầu tiên.
2. Bạn thứ `1` chuyền bóng cho bạn cách mình `k` bước theo chiều kim đồng hồ.
3. Người vừa nhận bóng chuyền tiếp cho bạn cách mình `2 * k` bước.
4. Người nhận tiếp theo chuyền cho bạn cách `3 * k` bước, và cứ thế: ở lượt thứ `i`, người đang giữ bóng chuyền đi `i * k` bước.
5. Trò chơi **kết thúc khi có người nhận bóng lần thứ hai**.

Những người **không hề nhận bóng** trong cả trò chơi là **kẻ thua (losers)**. Trả về danh sách kẻ thua **theo thứ tự tăng dần**.

| Ví dụ | n | k | Output | Giải thích |
|---|---|---|---|---|
| 1 | 5 | 2 | `[4, 5]` | `1 → 3 → 2 → 3`: bạn 3 nhận lần hai nên dừng. Chưa nhận: 4, 5 |
| 2 | 4 | 4 | `[2, 3, 4]` | bạn 1 chuyền 4 bước (đúng một vòng) nên lại tới chính mình, dừng ngay. Chưa nhận: 2, 3, 4 |

Constraints: `1 <= k <= n <= 50`.

In [1]:
# Ví dụ 1 tính tay: n = 5, k = 2 (đánh số 1..5)
n, k = 5, 2
pos = 1
print("lượt 0: bạn", pos, "nhận bóng (lần đầu)")
for i in range(1, 4):
    pos = (pos - 1 + i * k) % n + 1      # công thức chuyển 1-indexed sẽ được giải thích ở mục 3
    print(f"lượt {i}: chuyền {i * k} bước -> bạn {pos} nhận")

lượt 0: bạn 1 nhận bóng (lần đầu)
lượt 1: chuyền 2 bước -> bạn 3 nhận
lượt 2: chuyền 4 bước -> bạn 2 nhận
lượt 3: chuyền 6 bước -> bạn 3 nhận


## 3. Kiến thức nền

### 3.1. Đi vòng tròn bằng phép chia lấy dư `%`

Với `n` chỗ ngồi đánh số `0, 1, ..., n - 1`, đi thêm `s` bước từ vị trí `p` thì tới `(p + s) % n`. Phép `%` cho **số dư**, nên mỗi khi vượt quá `n - 1` ta "quay lại từ 0".

Ví dụ `n = 5`: từ vị trí `3` đi `4` bước: `3 + 4 = 7`, `7 % 5 = 2`.

In [2]:
n = 5
print((3 + 4) % n)      # 2
print((4 + 1) % n)      # 0  (từ chỗ cuối quay lại chỗ đầu)
print(12 % 5, 15 % 5)   # 2 0

2
0
2 0


### 3.2. Đánh số từ 1 hay từ 0?

Đề đánh số bạn từ `1` nhưng phép `%` hợp với đánh số từ `0` hơn. Cách làm gọn: **đổi sang 0-indexed khi tính toán**, lúc trả đáp án mới cộng `1`:

- bạn `1` ứng với vị trí `0`; bạn `i` ứng với vị trí `i - 1`;
- vị trí `p` ứng với bạn `p + 1`.

Nếu cứ giữ 1-indexed, công thức dễ sai vì `n % n = 0` mà "bạn 0" không tồn tại.

In [3]:
n = 5
# 1-indexed sai: bạn 5 đi thêm 1 bước phải tới bạn 1, nhưng (5 + 1) % 5 = 1 đúng, còn (4 + 1) % 5 = 0 là "bạn 0"?
print((4 + 1) % n)          # 0  -> không có bạn số 0
# 0-indexed đúng: vị trí 4 (bạn 5) đi 1 bước:
print((4 + 1) % n + 1)      # 1  -> bạn 1

0
1


### 3.3. Mảng đánh dấu `visited` bằng list `bool`

`[False] * n` tạo list gồm `n` giá trị `False`. Khi ai đó nhận bóng, ta gán `visited[p] = True`. Kiểm tra "đã nhận chưa" chỉ là đọc `visited[p]`, tốn `O(1)`.

In [4]:
n = 5
visited = [False] * n
visited[0] = True
visited[2] = True
print(visited)             # [True, False, True, False, False]
print(visited[2], visited[3])

[True, False, True, False, False]
True False


### 3.4. `set` (tập hợp): thêm, kiểm tra thuộc, hiệu hai tập

`set` là tập các phần tử **không trùng**. `s.add(x)` thêm; `x in s` kiểm tra thuộc (nhanh, trung bình `O(1)`); `a - b` là các phần tử có trong `a` mà không có trong `b`.

In [5]:
got = set()
got.add(1)
got.add(3)
got.add(3)                 # thêm lần hai không đổi gì
print(got, 3 in got, 4 in got)
everyone = set(range(1, 6))        # {1, 2, 3, 4, 5}
print(everyone - got)              # những người chưa nhận: {2, 4, 5}
print(sorted(everyone - got))      # sorted trả về list đã sắp tăng dần

{1, 3} True False
{2, 4, 5}
[2, 4, 5]


### 3.5. `while True` + `break`, và list comprehension

- `while True:` lặp mãi cho đến khi gặp `break`. Hợp với bài này vì ta không biết trước trò chơi kéo dài mấy lượt, chỉ biết "dừng khi có người nhận lần hai".
- List comprehension `[i + 1 for i in range(n) if not visited[i]]` gom các bạn (đổi sang số 1-indexed) mà `visited[i]` còn `False`, theo thứ tự `i` tăng dần nên **kết quả tự động đã sắp xếp**.

In [6]:
visited = [True, False, True, False, False]
losers = [i + 1 for i in range(len(visited)) if not visited[i]]
print(losers)       # [2, 4, 5]

count = 0
while True:
    count += 1
    if count == 3:
        break
print("dừng ở", count)

[2, 4, 5]
dừng ở 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đề chỉ ra **luật chuyền bóng rất cụ thể**, nên cách tự nhiên nhất là **làm đúng như luật**:

1. Đứng ở vị trí hiện tại, ghi "người này đã nhận bóng".
2. Tính người nhận kế tiếp: đi thêm `bước * k` chỗ, quay vòng bằng `% n`.
3. Tăng `bước` thêm 1.
4. Nếu người kế tiếp **đã nhận rồi** → dừng.

Tính tay `n = 5, k = 2` (0-indexed, vị trí 0 là bạn 1):

| lượt | đang ở | chuyền `lượt·k` bước | tới | đã nhận trước đó? |
|---|---|---|---|---|
| 1 | 0 | 2 | 2 | chưa |
| 2 | 2 | 4 | 6 % 5 = 1 | chưa |
| 3 | 1 | 6 | 7 % 5 = 2 | rồi (vị trí 2) → dừng |

Đã nhận: vị trí `0, 2, 1` tức bạn `1, 3, 2`; chưa nhận: bạn `4, 5`. Khớp ví dụ.

In [7]:
n, k = 5, 2
pos, step = 0, 1
seen_order = [0]
while True:
    pos = (pos + step * k) % n
    if pos in seen_order:
        print("dừng: vị trí", pos, "nhận lần 2")
        break
    seen_order.append(pos)
    step += 1
print("thứ tự nhận (0-indexed):", seen_order)

dừng: vị trí 2 nhận lần 2
thứ tự nhận (0-indexed): [0, 2, 1]


## 5. Cách 1 — Brute Force / baseline: mô phỏng với `list` và kiểm tra `in`

**Ý tưởng:** giữ một **danh sách** `got` các vị trí đã nhận bóng. Mỗi lượt kiểm tra `pos in got` (duyệt tuyến tính trong list). Cuối cùng, với từng bạn `1..n`, nếu không nằm trong `got` thì là kẻ thua.

Code này **cố tình đơn giản**: mỗi phép `in` trên list là `O(độ dài list)`.

In [8]:
from typing import List

def losers_list(n: int, k: int) -> List[int]:
    got = [0]                 # vị trí 0 (bạn 1) nhận bóng đầu tiên
    pos, step = 0, 1
    while True:
        pos = (pos + step * k) % n
        if pos in got:        # list: duyệt tuyến tính
            break
        got.append(pos)
        step += 1
    res = []
    for friend in range(1, n + 1):
        if (friend - 1) not in got:     # lại duyệt tuyến tính
            res.append(friend)
    return res

print(losers_list(5, 2))   # [4, 5]
print(losers_list(4, 4))   # [2, 3, 4]

[4, 5]
[2, 3, 4]


**Độ phức tạp:** trò chơi dừng sau tối đa `n` lượt (xem mục 10). Mỗi lượt `in got` tốn `O(n)` nên phần mô phỏng `O(n²)`; phần lọc kết quả cũng `O(n²)`. Time `O(n²)`, Space `O(n)`. Với `n ≤ 50` vẫn chạy ngay lập tức, nhưng tư duy này **không mở rộng** khi `n` lớn.

**Nút thắt:** việc hỏi "người này đã nhận chưa?" bị lặp lại nhiều lần và mỗi lần phải quét cả danh sách. Cần một cấu trúc tra cứu `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Câu hỏi "đã nhận chưa?" chỉ cần trả lời **có/không** cho một vị trí `0..n-1` ⇒ dùng **mảng `visited` có `n` ô**, tra cứu `visited[pos]` tốn `O(1)`.

Hơn nữa có hai sự thật giúp mọi thứ gọn:

- Mỗi lượt đánh dấu **một** vị trí mới; chỉ có `n` vị trí nên sau tối đa `n` lượt phải gặp lại một vị trí đã đánh dấu ⇒ trò chơi dừng sau **≤ n lượt**, nên cả mô phỏng là `O(n)`.
- Danh sách kẻ thua = các chỉ số còn `False` trong `visited`, đọc theo thứ tự chỉ số tăng dần ⇒ **không cần sắp xếp** thêm.

In [9]:
n, k = 5, 2
visited = [False] * n
pos, step = 0, 1
while not visited[pos]:
    visited[pos] = True
    print(f"bạn {pos + 1} nhận bóng; visited = {visited}")
    pos = (pos + step * k) % n
    step += 1
print("bạn", pos + 1, "nhận lần hai -> dừng")

bạn 1 nhận bóng; visited = [True, False, False, False, False]
bạn 3 nhận bóng; visited = [True, False, True, False, False]
bạn 2 nhận bóng; visited = [True, True, True, False, False]
bạn 3 nhận lần hai -> dừng


## 7. Các cách tiếp theo

Ta sẽ xem thêm 3 cách (tổng cộng 4 cách): (2) mảng `visited` — cách chuẩn, (3) `set` + hiệu tập hợp, (4) công thức số tam giác tính thẳng vị trí ở lượt `t`.

### Cách 2 — Mô phỏng với mảng `visited`

Giải thích từng đoạn:

- `visited = [False] * n`: chưa ai nhận bóng.
- `pos = 0`: bạn 1 (vị trí 0) cầm bóng; `step = 1`: lượt chuyền đầu là `1 * k` bước.
- `while not visited[pos]`: **chừng nào** người đang cầm bóng chưa từng nhận thì ghi nhận cho họ.
- `pos = (pos + step * k) % n; step += 1`: chuyền tới người kế, tăng bước nhảy.
- Thoát vòng lặp nghĩa là `visited[pos]` đã `True` → có người nhận lần hai.
- `[i + 1 for i in range(n) if not visited[i]]`: gom kẻ thua theo thứ tự tăng dần.

In [10]:
def losers_visited(n: int, k: int) -> List[int]:
    visited = [False] * n
    pos, step = 0, 1
    while not visited[pos]:
        visited[pos] = True
        pos = (pos + step * k) % n
        step += 1
    return [i + 1 for i in range(n) if not visited[i]]

print(losers_visited(5, 2), losers_visited(4, 4))

[4, 5] [2, 3, 4]


In [11]:
from typing import List

class SolutionVisited:
    def circularGameLosers(self, n: int, k: int) -> List[int]:
        visited = [False] * n
        pos, step = 0, 1
        while not visited[pos]:
            visited[pos] = True
            pos = (pos + step * k) % n
            step += 1
        return [i + 1 for i in range(n) if not visited[i]]

print(SolutionVisited().circularGameLosers(5, 2))

[4, 5]


**Độ phức tạp:** tối đa `n` lượt, mỗi lượt `O(1)`; lọc kết quả `O(n)`. Time `O(n)`, Space `O(n)` cho mảng `visited`.

### Cách 3 — `set` các người đã nhận + hiệu hai tập hợp

Thay mảng `bool` bằng `set` chứa vị trí đã nhận. Ưu điểm: viết "người chưa nhận" rất tự nhiên bằng phép trừ tập hợp `tất_cả - đã_nhận`. Lưu ý `set` **không có thứ tự** nên phải `sorted(...)` ở cuối để thoả "tăng dần".

Giải thích từng đoạn:

- `got = set()`, vòng `while pos not in got` thay cho `while not visited[pos]`.
- `everyone = set(range(1, n + 1))`, và ta lưu người nhận bằng số 1-indexed `pos + 1` để trừ được ngay.
- `sorted(everyone - got)` ra list tăng dần.

In [12]:
def losers_set(n: int, k: int) -> List[int]:
    got = set()
    pos, step = 0, 1
    while pos + 1 not in got:
        got.add(pos + 1)
        pos = (pos + step * k) % n
        step += 1
    everyone = set(range(1, n + 1))
    return sorted(everyone - got)

print(losers_set(5, 2), losers_set(4, 4))

[4, 5] [2, 3, 4]


In [13]:
from typing import List

class SolutionSet:
    def circularGameLosers(self, n: int, k: int) -> List[int]:
        got = set()
        pos, step = 0, 1
        while pos + 1 not in got:
            got.add(pos + 1)
            pos = (pos + step * k) % n
            step += 1
        return sorted(set(range(1, n + 1)) - got)

print(SolutionSet().circularGameLosers(4, 4))

[2, 3, 4]


**Độ phức tạp:** mô phỏng `O(n)`; `sorted` trên tối đa `n` phần tử là `O(n log n)`. Time `O(n log n)`, Space `O(n)`. Chậm hơn một chút so với Cách 2 về lý thuyết (do `sorted`), nhưng với `n ≤ 50` không đáng kể.

### Cách 4 — Công thức số tam giác: vị trí ở lượt `t` là `k·t(t+1)/2 mod n`

Tổng các bước nhảy sau `t` lượt là `k + 2k + ... + tk = k·(1 + 2 + ... + t) = k·t(t+1)/2` (số tam giác nhân `k`). Vì bắt đầu từ vị trí `0`, người cầm bóng sau `t` lượt ở vị trí:

`pos(t) = (k * t * (t + 1) // 2) % n`

Không cần biến `pos` tích luỹ; ta chỉ cần lặp `t = 0, 1, 2, ...` và tính thẳng. (Chú ý dùng `//` an toàn vì `t(t+1)` luôn chẵn.)

In [14]:
n, k = 5, 2
for t in range(4):
    print(t, (k * t * (t + 1) // 2) % n)    # 0 2 1 2

0 0
1 2
2 1
3 2


In [15]:
from typing import List

class SolutionTriangular:
    def circularGameLosers(self, n: int, k: int) -> List[int]:
        visited = [False] * n
        t = 0
        while True:
            pos = (k * t * (t + 1) // 2) % n
            if visited[pos]:
                break
            visited[pos] = True
            t += 1
        return [i + 1 for i in range(n) if not visited[i]]

print(SolutionTriangular().circularGameLosers(5, 2))

[4, 5]


**Độ phức tạp:** như Cách 2: Time `O(n)`, Space `O(n)`. Số trong công thức tối đa cỡ `k·n²/2 ≈ 50·2500/2` rất nhỏ; trong ngôn ngữ có tràn số, đây là chỗ cần để ý (Python thì không lo).

## 8. TRACE từng bước

Trace Cách 2 với `n = 5, k = 2`:

| vòng | `pos` (0-idx) | `step` | `visited[pos]` trước | hành động | `pos` mới |
|---|---|---|---|---|---|
| 1 | 0 | 1 | False | đánh dấu 0, `pos=(0+1·2)%5` | 2 |
| 2 | 2 | 2 | False | đánh dấu 2, `pos=(2+2·2)%5` | 1 |
| 3 | 1 | 3 | False | đánh dấu 1, `pos=(1+3·2)%5` | 2 |
| 4 | 2 | 4 | **True** | dừng | — |

Kết quả `visited = [T, T, T, F, F]` → kẻ thua là vị trí 3, 4 → bạn `4, 5`.

In [16]:
def trace_visited(n, k):
    visited = [False] * n
    pos, step = 0, 1
    vong = 0
    while True:
        vong += 1
        print(f"vòng {vong}: pos={pos} (bạn {pos + 1}), step={step}, visited[pos]={visited[pos]}")
        if visited[pos]:
            print("  -> đã nhận rồi, DỪNG")
            break
        visited[pos] = True
        nxt = (pos + step * k) % n
        print(f"  -> đánh dấu; chuyền {step * k} bước tới {nxt}; visited={visited}")
        pos, step = nxt, step + 1
    ans = [i + 1 for i in range(n) if not visited[i]]
    print("kẻ thua:", ans)
    return ans

trace_visited(5, 2)

vòng 1: pos=0 (bạn 1), step=1, visited[pos]=False
  -> đánh dấu; chuyền 2 bước tới 2; visited=[True, False, False, False, False]
vòng 2: pos=2 (bạn 3), step=2, visited[pos]=False
  -> đánh dấu; chuyền 4 bước tới 1; visited=[True, False, True, False, False]
vòng 3: pos=1 (bạn 2), step=3, visited[pos]=False
  -> đánh dấu; chuyền 6 bước tới 2; visited=[True, True, True, False, False]
vòng 4: pos=2 (bạn 3), step=4, visited[pos]=True
  -> đã nhận rồi, DỪNG
kẻ thua: [4, 5]


[4, 5]

In [17]:
# Trace thứ hai: trò chơi dừng ngay lượt đầu (k = n)
trace_visited(4, 4)
# Và một trường hợp dài: n = 7, k = 3
trace_visited(7, 3)

vòng 1: pos=0 (bạn 1), step=1, visited[pos]=False
  -> đánh dấu; chuyền 4 bước tới 0; visited=[True, False, False, False]
vòng 2: pos=0 (bạn 1), step=2, visited[pos]=True
  -> đã nhận rồi, DỪNG
kẻ thua: [2, 3, 4]
vòng 1: pos=0 (bạn 1), step=1, visited[pos]=False
  -> đánh dấu; chuyền 3 bước tới 3; visited=[True, False, False, False, False, False, False]
vòng 2: pos=3 (bạn 4), step=2, visited[pos]=False
  -> đánh dấu; chuyền 6 bước tới 2; visited=[True, False, False, True, False, False, False]
vòng 3: pos=2 (bạn 3), step=3, visited[pos]=False
  -> đánh dấu; chuyền 9 bước tới 4; visited=[True, False, True, True, False, False, False]
vòng 4: pos=4 (bạn 5), step=4, visited[pos]=False
  -> đánh dấu; chuyền 12 bước tới 2; visited=[True, False, True, True, True, False, False]
vòng 5: pos=2 (bạn 3), step=5, visited[pos]=True
  -> đã nhận rồi, DỪNG
kẻ thua: [2, 6, 7]


[2, 6, 7]

## 9. Code nộp LeetCode

Dùng Cách 2 (mảng `visited`), đúng class `Solution` và method `circularGameLosers`.

In [18]:
from typing import List

class Solution:
    def circularGameLosers(self, n: int, k: int) -> List[int]:
        visited = [False] * n
        pos, step = 0, 1
        while not visited[pos]:
            visited[pos] = True
            pos = (pos + step * k) % n
            step += 1
        return [i + 1 for i in range(n) if not visited[i]]

In [19]:
print(Solution().circularGameLosers(5, 2))    # [4, 5]
print(Solution().circularGameLosers(4, 4))    # [2, 3, 4]

[4, 5]
[2, 3, 4]


## 10. Vì sao thuật toán đúng

**Mô phỏng đúng luật.** Ở lượt `step`, người cầm bóng chuyền `step * k` bước theo chiều kim đồng hồ ⇒ vị trí mới là `(pos + step * k) % n`. Bắt đầu `pos = 0` (bạn 1) và `step = 1` đúng như đề.

**Điều kiện dừng đúng.** Vòng `while not visited[pos]` kiểm tra người **sắp nhận** bóng đã nhận trước đó chưa. Nếu rồi thì đó chính là "nhận bóng lần thứ hai" ⇒ dừng ngay, **không** đánh dấu thêm. Người cầm bóng đầu tiên cũng được tính là đã nhận (ta đánh dấu `pos = 0` ngay vòng đầu).

**Dừng sau tối đa `n` lượt.** Mỗi vòng lặp đánh dấu một ô `False` thành `True`. Chỉ có `n` ô, nên sau tối đa `n` vòng mọi ô đã `True` và lần kiểm tra kế tiếp phải thấy `True` ⇒ không lặp vô hạn (chỉ trong trường hợp xấu nhất mới đánh dấu hết `n` người).

**Kết quả đúng.** Kẻ thua = người chưa bao giờ nhận = ô còn `False`. Duyệt `i` tăng dần nên output tăng dần. Cộng `1` để đổi sang đánh số 1-indexed của đề.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số người.

| Cách | Time | Space |
|---|---|---|
| 1. list + `in` | `O(n²)` | `O(n)` |
| 2. mảng `visited` | `O(n)` | `O(n)` |
| 3. `set` + hiệu tập | `O(n log n)` | `O(n)` |
| 4. công thức tam giác | `O(n)` | `O(n)` |

Mỗi cách chạy nhiều nhất `n` lượt chuyền (mục 10); riêng Cách 1 có thêm chi phí `O(n)` cho mỗi phép `in` trên list, Cách 3 thêm bước `sorted`. Bộ nhớ chủ yếu là mảng/tập `visited` kích thước tối đa `n`. `k` không ảnh hưởng độ phức tạp vì ta luôn lấy `% n`.

## 12. Bộ test

### 12.1. Ví dụ của đề

Ta gom mọi cách giải vào một danh sách rồi cho chạy cùng bộ test.

In [20]:
sols = [SolutionVisited(), SolutionSet(), SolutionTriangular(), Solution()]
funcs = [losers_list, losers_visited, losers_set] + [s.circularGameLosers for s in sols]

for f in funcs:
    assert f(5, 2) == [4, 5]
    assert f(4, 4) == [2, 3, 4]
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case

- `n = 1, k = 1`: chỉ có một bạn; chuyền `1` bước vẫn về chính mình ⇒ dừng, không ai thua `[]`.
- `n = 2, k = 1`: `1 → 2`, rồi bạn 2 chuyền `2` bước (đủ một vòng) nên về chính mình ⇒ dừng, `[]`.
- `n = 3, k = 1`: `1 → 2 → (2+2 = 4 → 1)`: bạn 1 nhận lần hai ⇒ thua: bạn 3.
- `k = n` (mọi `n`): bước chuyền `i·n` luôn là bội của `n` nên bóng đứng yên ở bạn 1 ⇒ kẻ thua `[2, ..., n]`.
- `n = 50, k = 50` và `n = 50, k = 1` (giá trị lớn nhất).

In [21]:
for f in funcs:
    assert f(1, 1) == []
    assert f(2, 1) == []
    assert f(3, 1) == [3]
    for n in (1, 2, 7, 50):
        assert f(n, n) == list(range(2, n + 1))
    assert isinstance(f(50, 1), list)
print("Edge case OK")
print("n=50, k=1 ->", Solution().circularGameLosers(50, 1))

Edge case OK
n=50, k=1 -> [3, 5, 8, 9, 10, 12, 13, 14, 15, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50]


### 12.3. Phản ví dụ cho các cách làm sai thường gặp

Ví dụ: một cách làm sai là "bước nhảy **không tăng**" (luôn đi `k`). Với `n = 5, k = 2` nó cho kết quả khác đề (xem mục 13). Ta chứng minh nó sai bằng một `assert` ở đó.

### 12.4. Đối chiếu nhiều input ngẫu nhiên (có seed)

So sánh 4 cách + một **oracle viết khác hẳn**: dùng `dict` đếm số lần nhận bóng và dừng khi một người đạt `2`, mô phỏng thẳng theo câu chữ của đề, với toàn bộ miền hợp lệ `1 <= k <= n <= 50`.

In [22]:
import random

def oracle(n, k):
    count = {}
    pos = 1                         # 1-indexed, giữ y nguyên theo đề
    count[pos] = 1
    i = 1
    while True:
        pos = (pos - 1 + i * k) % n + 1
        count[pos] = count.get(pos, 0) + 1
        if count[pos] == 2:
            break
        i += 1
    return [f for f in range(1, n + 1) if f not in count]

random.seed(2682)
# 1) toàn bộ miền hợp lệ (n <= 50 nên vét hết được: ~1275 cặp)
total = 0
for n in range(1, 51):
    for k in range(1, n + 1):
        want = oracle(n, k)
        for f in funcs:
            assert f(n, k) == want, (n, k, f)
        total += 1
print("đã kiểm", total, "cặp (n, k) trong toàn miền, 7 cách khớp oracle")

# 2) thêm 300 cặp ngẫu nhiên có seed (để minh hoạ cách viết test ngẫu nhiên)
for _ in range(300):
    n = random.randint(1, 50)
    k = random.randint(1, n)
    assert Solution().circularGameLosers(n, k) == oracle(n, k)
print("test ngẫu nhiên OK")

đã kiểm 1275 cặp (n, k) trong toàn miền, 7 cách khớp oracle
test ngẫu nhiên OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Bước nhảy không tăng (luôn nhảy `k`)

Đề viết "`k`, rồi `2k`, rồi `3k`...". Người mới đọc lướt và chỉ nhảy `k` mỗi lần.

In [23]:
from typing import List

def sai_buoc_khong_tang(n: int, k: int) -> List[int]:
    visited = [False] * n
    pos = 0
    while not visited[pos]:
        visited[pos] = True
        pos = (pos + k) % n          # SAI: thiếu step
    return [i + 1 for i in range(n) if not visited[i]]

print("sai :", sai_buoc_khong_tang(5, 2))        # [] vì 0 -> 2 -> 4 -> 1 -> 3 -> 0 ghé thăm cả 5 bạn
print("đúng:", Solution().circularGameLosers(5, 2))   # [4, 5]
assert sai_buoc_khong_tang(5, 2) != Solution().circularGameLosers(5, 2)

sai : []
đúng: [4, 5]


**Vì sao sai:** khi `k` nguyên tố cùng nhau với `n`, bước cố định `k` đi qua **tất cả** mọi người trước khi lặp, nên không ai thua. Còn luật thật tăng dần `k, 2k, 3k...` nên vị trí lặp rất sớm.

### Lỗi 2: Lệch chỉ số 1-indexed / 0-indexed

In [24]:
def sai_lech_chi_so(n: int, k: int) -> List[int]:
    visited = [False] * (n + 1)      # dùng 1-indexed để "khỏi đổi"
    pos, step = 1, 1
    while not visited[pos]:
        visited[pos] = True
        pos = (pos + step * k) % n   # SAI: kết quả % n có thể bằng 0 (không có bạn 0)
        step += 1
    return [i for i in range(1, n + 1) if not visited[i]]

bad = [(n, k) for n in range(1, 51) for k in range(1, n + 1)
       if sai_lech_chi_so(n, k) != Solution().circularGameLosers(n, k)]
print("số cặp (n, k) bị sai:", len(bad), "ví dụ:", bad[:3])
n0, k0 = bad[0]
print("sai :", sai_lech_chi_so(n0, k0))
print("đúng:", Solution().circularGameLosers(n0, k0))
assert len(bad) > 0

số cặp (n, k) bị sai: 269 ví dụ: [(2, 1), (3, 2), (4, 1)]
sai : [2]
đúng: []


**Vì sao sai:** `% n` cho số trong `0..n-1`, trong đó có `0` — một "bạn số 0" không tồn tại; còn bạn `n` thì không bao giờ tới được. Muốn 1-indexed phải dùng `(pos - 1 + step*k) % n + 1`; cách gọn hơn là tính 0-indexed rồi cộng `1` khi xuất kết quả.

### Lỗi 3: Quên đánh dấu bạn 1 (người cầm bóng đầu tiên)

In [25]:
def sai_quen_danh_dau_bat_dau(n: int, k: int) -> List[int]:
    visited = [False] * n
    pos, step = 0, 1
    # SAI: quên visited[0] = True cho người cầm bóng đầu tiên
    while True:
        pos = (pos + step * k) % n
        if visited[pos]:
            break
        visited[pos] = True
        step += 1
    return [i + 1 for i in range(n) if not visited[i]]

print("sai :", sai_quen_danh_dau_bat_dau(5, 2))     # bạn 1 bị tính là chưa nhận
print("đúng:", Solution().circularGameLosers(5, 2))
assert sai_quen_danh_dau_bat_dau(5, 2) != Solution().circularGameLosers(5, 2)

sai : [1, 4, 5]
đúng: [4, 5]


**Vì sao sai:** bạn 1 đã nhận bóng ngay từ đầu. Nếu không đánh dấu, ở ví dụ 1 (`n=5, k=2`) bạn 1 bị tính là chưa nhận bóng nên lọt vào danh sách kẻ thua, và trò chơi dừng sai lúc (bóng quay về bạn 1 mà vẫn coi là "lần đầu").

### Lỗi 4: Quên sắp xếp khi dùng `set`

Với `set`, thứ tự duyệt không đảm bảo tăng dần; luôn `sorted(...)` hoặc dùng mảng `visited` (tự có thứ tự).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có luật "làm từng bước cho đến khi gặp điều kiện dừng" ⇒ **Simulation**. Cứ dịch luật thành vòng lặp; đừng tìm công thức khi chưa cần.
- "Đi vòng tròn", "quay lại đầu" ⇒ phép `% n` (nhớ đổi sang **0-indexed**).
- "Ai đã/chưa làm điều gì" ⇒ mảng `visited` hoặc `set`.
- Điều kiện dừng kiểu "**lần thứ hai**" thường được đảm bảo bởi **nguyên lý chuồng bồ câu** (pigeonhole): `n` ô thì tối đa `n` lượt mới lặp. Biết điều này cho ta cận thời gian `O(n)`.
- Constraints nhỏ (`n ≤ 50`) nghĩa là cách nào đúng cũng được, nhưng nên viết cách `O(n)` cho gọn.
- Khi phải trả mảng "tăng dần", hãy duyệt chỉ số tăng dần thay vì `sort` ở cuối.

## 15. Biến thể của bài

1. **Đếm số người nhận bóng:** `sum(visited)` — hoặc `n - len(losers)`.
2. **Số lượt chuyền** cho đến khi trò chơi dừng (độ dài chu trình + phần "đuôi").
3. **Bước nhảy khác:** lượt `i` chuyền `i²` bước, hoặc `k·2^i` bước. Chỉ đổi dòng tính `pos`, phần còn lại giữ nguyên.
4. **Người bắt đầu là bạn `s`** thay vì bạn 1.
5. **Chuyền ngược chiều kim đồng hồ:** dùng `(pos - step * k) % n` — Python cho kết quả không âm với `%`.
6. **Josephus** (LeetCode 1823 Find the Winner of the Circular Game) cũng là trò chơi vòng tròn, nhưng người bị loại sẽ bị bỏ khỏi vòng — cách xử lý khác hẳn.

In [26]:
# Biến thể: đếm số lượt chuyền và số người nhận bóng
def thong_ke(n, k):
    visited = [False] * n
    pos, step, luot = 0, 1, 0
    while not visited[pos]:
        visited[pos] = True
        pos = (pos + step * k) % n
        step += 1
        luot += 1
    return luot, sum(visited)

print(thong_ke(5, 2))     # (3, 3): 3 người nhận bóng, 3 lượt chuyền trước khi gặp lại
print(thong_ke(4, 4))     # (1, 1)

(3, 3)
(1, 1)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. list + `in` | mô phỏng, lưu list vị trí đã nhận | `O(n²)` | `O(n)` | hiểu đề, `n` rất nhỏ |
| 2. mảng `visited` | mô phỏng, tra cứu `O(1)` | `O(n)` | `O(n)` | **cách nộp bài khuyến nghị** |
| 3. `set` + hiệu tập | mô phỏng, tập hợp đã nhận | `O(n log n)` | `O(n)` | thích phong cách tập hợp, code ngắn |
| 4. công thức tam giác | tính thẳng `k·t(t+1)/2 mod n` | `O(n)` | `O(n)` | hiểu cấu trúc toán; biến thể lượt `t` tuỳ ý |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(7 + 3) % 4`, `(0 - 1) % 5`, `sorted({3, 1, 2} - {2})`, và `[i + 1 for i in range(4) if i % 2 == 0]`.

**Bài 2 (tính tay).** Với `n = 6, k = 2`, viết dãy vị trí (0-indexed) mà bóng đi qua cho tới khi có người nhận lần hai, rồi suy ra kẻ thua.

**Bài 3 (điền TODO).** Hoàn thành `losers_todo(n, k)` bằng mảng `visited`.

In [27]:
def losers_todo(n, k):
    # TODO 1: visited = [False] * n; pos, step = 0, 1
    # TODO 2: while not visited[pos]: đánh dấu, pos = (pos + step * k) % n, step += 1
    # TODO 3: return [i + 1 for i in range(n) if not visited[i]]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `losers_triangular(n, k)` bằng công thức `(k * t * (t + 1) // 2) % n` mà không nhìn Cách 4. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `rounds(n, k)` trả về **số lượt chuyền** đã diễn ra khi trò chơi dừng. (b) Viết `losers_from(n, k, s)` với bạn `s` cầm bóng đầu tiên (`1 <= s <= n`), kết quả vẫn tăng dần.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `10 % 4 = 2`; Python cho `(-1) % 5 = 4`; `{3,1,2} - {2} = {1,3}`; vị trí chẵn là `0, 2` nên list `[1, 3]`. Bài 2: bắt đầu `0`, bước nhảy `2, 4, 6, ...` mod `6`.
- **Mức 2 (vừa):** Bài 3: ba TODO chính là Cách 2. Bài 4: lặp `t = 0, 1, ...`; `pos = (k*t*(t+1)//2) % n`; dừng khi `visited[pos]`.
- **Mức 3 (sâu):** Bài 5(a): đếm số lần thực hiện thân vòng `while`. Bài 5(b): đổi `pos = s - 1` ban đầu; còn lại không đổi.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
assert (7 + 3) % 4 == 2
assert (0 - 1) % 5 == 4
assert sorted({3, 1, 2} - {2}) == [1, 3]
assert [i + 1 for i in range(4) if i % 2 == 0] == [1, 3]

# Bài 2: n = 6, k = 2 -> 0, (0+2)%6=2, (2+4)%6=0 (đã nhận) => dừng
order = [0]
pos, step = 0, 1
while True:
    pos = (pos + step * 2) % 6
    if pos in order:
        break
    order.append(pos)
    step += 1
assert order == [0, 2]
assert [i + 1 for i in range(6) if i not in order] == [2, 4, 5, 6]

# Bài 3
def losers_todo_ans(n, k):
    visited = [False] * n
    pos, step = 0, 1
    while not visited[pos]:
        visited[pos] = True
        pos = (pos + step * k) % n
        step += 1
    return [i + 1 for i in range(n) if not visited[i]]

assert losers_todo_ans(5, 2) == [4, 5]
assert losers_todo_ans(4, 4) == [2, 3, 4]

# Bài 4
def losers_triangular(n, k):
    visited = [False] * n
    t = 0
    while True:
        pos = (k * t * (t + 1) // 2) % n
        if visited[pos]:
            break
        visited[pos] = True
        t += 1
    return [i + 1 for i in range(n) if not visited[i]]

assert losers_triangular(5, 2) == [4, 5]
assert losers_triangular(4, 4) == [2, 3, 4]

# Bài 5
def rounds(n, k):
    visited = [False] * n
    pos, step, cnt = 0, 1, 0
    while not visited[pos]:
        visited[pos] = True
        pos = (pos + step * k) % n
        step += 1
        cnt += 1
    return cnt

assert rounds(5, 2) == 3 and rounds(4, 4) == 1

def losers_from(n, k, s):
    visited = [False] * n
    pos, step = s - 1, 1
    while not visited[pos]:
        visited[pos] = True
        pos = (pos + step * k) % n
        step += 1
    return [i + 1 for i in range(n) if not visited[i]]

assert losers_from(5, 2, 1) == [4, 5]
assert losers_from(5, 2, 2) == [1, 5]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mô phỏng + mảng `visited`)

```python
from typing import List

class Solution:
    def circularGameLosers(self, n: int, k: int) -> List[int]:
        visited = [False] * n
        pos, step = 0, 1
        while not visited[pos]:
            visited[pos] = True
            pos = (pos + step * k) % n
            step += 1
        return [i + 1 for i in range(n) if not visited[i]]
```

**Checklist**

- [ ] Đổi sang 0-indexed khi tính, cộng `1` khi trả kết quả?
- [ ] Bước nhảy **tăng dần** (`step * k`, rồi `step += 1`)?
- [ ] Đánh dấu bạn 1 ngay từ đầu (vòng đầu đánh dấu `pos = 0`)?
- [ ] Dừng khi người **sắp nhận** đã nhận rồi (không đánh dấu thêm)?
- [ ] Kết quả theo thứ tự tăng dần?
- [ ] Đã thử `n = 1`, `k = n`, và `n = 50`?
- [ ] Signature đúng `circularGameLosers(self, n: int, k: int) -> List[int]` và có `from typing import List`?